# 02 - Modelo de Detección de Fraude
## FraudIA Claims

Este notebook entrena y evalúa los modelos de ML para detección de fraude:
- Random Forest (supervisado)
- Isolation Forest (anomalías)
- Score híbrido (reglas + ML + anomalías)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingestion.load_data import load_all_from_directory
from src.features.build_features import build_all_features, get_feature_columns
from src.rules.fraud_rules import apply_rules, get_rules_summary
from src.nlp.text_analysis import get_similarity_scores_by_id
from src.models.fraud_model import (
    train_supervised_model, train_anomaly_model,
    compute_hybrid_score, predict_fraud_probability
)

print('Módulos cargados')

## 1. Carga y Preparación de Datos

In [ ]:
datasets = load_all_from_directory('data/synthetic')
df = build_all_features(datasets)
print(f'Dataset con features: {df.shape}')
print(f'Features numéricas: {len(get_feature_columns(df))}')

## 2. Análisis NLP y Reglas de Negocio

In [ ]:
similarity_scores = get_similarity_scores_by_id(df)
high_sim = sum(1 for v in similarity_scores.values() if v >= 0.70)
print(f'Pares con similitud >= 70%: {high_sim}')

In [ ]:
df_rules = apply_rules(df, similarity_scores)
rules_summary = get_rules_summary(df_rules)
print('Resumen de Reglas:')
for k, v in rules_summary.items():
    print(f'  {k}: {v}')

## 3. Modelo Supervisado (Random Forest)

In [ ]:
feature_cols = get_feature_columns(df_rules)
print(f'Features para modelo: {len(feature_cols)}')
print(feature_cols)

In [ ]:
ml_results = train_supervised_model(df_rules, feature_cols)

print(f"AUC-ROC: {ml_results['auc_roc']}")
print(f"CV AUC Mean: {ml_results['cv_auc_mean']} ± {ml_results['cv_auc_std']}")
print(f"\nReporte de Clasificación:")
report = ml_results['classification_report']
for cls in ['0', '1']:
    if cls in report:
        print(f"  Clase {cls}: P={report[cls]['precision']:.3f} R={report[cls]['recall']:.3f} F1={report[cls]['f1-score']:.3f}")

In [ ]:
importances = pd.DataFrame(ml_results['feature_importance'])
top_n = importances.head(15)

plt.figure(figsize=(10, 8))
plt.barh(top_n['feature'][::-1], top_n['importance'][::-1], color='#3b82f6')
plt.title('Top 15 Features más Importantes (Random Forest)')
plt.xlabel('Importancia')
plt.tight_layout()
plt.savefig('data/processed/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cm = np.array(ml_results['confusion_matrix'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Fraude', 'Fraude'],
            yticklabels=['No Fraude', 'Fraude'])
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('data/processed/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Detección de Anomalías (Isolation Forest)

In [ ]:
anomaly_results = train_anomaly_model(df_rules, feature_cols)
print(f"Anomalías detectadas: {anomaly_results['n_anomalies']} de {anomaly_results['n_total']}")
print(f"Tasa de contaminación: {anomaly_results['contamination']}")

## 5. Score Híbrido

In [ ]:
df_rules['ml_fraud_probability'] = predict_fraud_probability(
    df_rules, ml_results['model'], ml_results['scaler'], ml_results['feature_cols']
)
df_rules['anomaly_score'] = anomaly_results['anomaly_scores']

df_scored = compute_hybrid_score(df_rules)
print(f"\nDistribución Semáforo Final:")
print(df_scored['semaforo_final'].value_counts())
print(f"\nScore Híbrido - Estadísticas:")
print(df_scored['score_hibrido'].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = {'Rojo': '#ef4444', 'Amarillo': '#f59e0b', 'Verde': '#22c55e'}
counts = df_scored['semaforo_final'].value_counts()
axes[0].pie(counts.values, labels=counts.index, colors=[colors.get(l, '#666') for l in counts.index], autopct='%1.1f%%')
axes[0].set_title('Distribución Semáforo')

df_scored['score_hibrido'].hist(bins=50, ax=axes[1], color='#3b82f6', edgecolor='white')
axes[1].axvline(26, color='#f59e0b', linestyle='--', label='Umbral Amarillo')
axes[1].axvline(56, color='#ef4444', linestyle='--', label='Umbral Rojo')
axes[1].set_title('Distribución Score Híbrido')
axes[1].legend()

for sem, color in colors.items():
    subset = df_scored[df_scored['semaforo_final'] == sem]
    axes[2].scatter(subset['monto_reclamado'], subset['score_hibrido'], c=color, label=sem, alpha=0.5, s=20)
axes[2].set_xlabel('Monto Reclamado')
axes[2].set_ylabel('Score Híbrido')
axes[2].set_title('Score vs Monto por Semáforo')
axes[2].legend()

plt.tight_layout()
plt.savefig('data/processed/score_hibrido.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_scored.to_csv('data/processed/siniestros_scored.csv', index=False, encoding='utf-8-sig')
print('Dataset con scores exportado a data/processed/siniestros_scored.csv')